# Generation

This notebook uses the collection stored in Milvus by the Task 2 notebook, you must run that notebook before this one.

The final step in a Retrieval-Augmented Generation (RAG) pipeline typically involves generating the final response or output based on the retrieved information and the query. Here’s a breakdown of the entire process and where the final step fits in:

Query Processing: A user's query is first processed, often by an encoder or a model that can understand the context and intent behind the query.

Document Retrieval: Based on the query, the system retrieves relevant documents, passages, or pieces of information from a knowledge base or corpus. This is usually done using a retrieval model (e.g., a dense retriever like Dense Retriever or BM25 for traditional search).

Contextualization (Prompt Engineering): The retrieved documents or passages are then combined with the original query, and potentially encoded into a new context. This step ensures that the relevant information is directly aligned with the query to help generate a more informed response.

Response Generation (Final Step): The final step is the generation phase, where a generative model (e.g., a transformer model like Llama or GPT) takes the combined query and retrieved context to produce a coherent and contextually relevant response. This step utilizes both the original query and the additional context from the retrieved documents to craft a response that answers the query effectively.

In short, the final step in a RAG pipeline is generating the final output or answer, which leverages both the query and the retrieved knowledge to create an informative, accurate, and contextually relevant response. This final generation is typically handled by a language model (e.g. `meta/llama-3.3-70b-instruct`)  that synthesizes information from the retrieval stage and ensures that the output aligns well with the user's intent.

<img src="https://blogs.nvidia.com/wp-content/uploads/2024/11/ragexplainer131-960x1143.png" width="600" />


We need to identify a question we would like to ask. To generate a meaningful answer the query must have an answer in the corpus. Otherwise, you may find yourself getting an answer that was hallucinated, or incorrect. This is why loading the corpus of data you want to ask questions about is so important. You need to make sure that there are answers in your corpus for the types of questions you want to ask. In this case we are working with the BEIR - Natural Questions dataset. This dataset was created by google, it consists of wikipedia information and related to real questions asked by users. Some of the questions asked are:  
- "when are hops added to the brewing process?" 
- "where is the world s largest ice sheet located today?" 
- "where is blood pumped after it leaves the right ventricle?" 
- "who is the voice of tony the tiger?" 
- "where does the energy in a nuclear explosion come from?"

In [10]:
query = "what color is the sky?"

In [11]:
from openai import OpenAI

api_key = "$API_KEY_REQUIRED_IF_EXECUTING_OUTSIDE_NGC"
openai_client = OpenAI(
  base_url = "http://embedding:8000/v1",
  api_key = api_key
)

In [12]:
import os

def get_answer(
    query,
    endpoint:str = "https://integrate.api.nvidia.com/v1", 
    model_name: str = "meta/llama-3.3-70b-instruct",
    api_key: str = None, 
):
    api_key = api_key or os.environ.get("NVIDIA_API_KEY", None)
    api_key = "<https://build.nvidia.com/>"
    client = OpenAI(
      base_url = endpoint,
      api_key = api_key
    )
    
    completion = client.chat.completions.create(
      model=model_name,
      messages=[{"role":"user","content":f"{query}"}],
      temperature=0.2,
      top_p=0.7,
      max_tokens=1024,
      stream=True
    )
    
    for chunk in completion:
      if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

get_answer(query)

The color of the sky can vary depending on the time of day, atmospheric conditions, and the observer's location. Here are some common colors the sky can appear:

1. **Blue**: During the daytime, the sky typically appears blue due to a phenomenon called Rayleigh scattering, where shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths by the Earth's atmosphere.
2. **Red/Pink/Orange**: During sunrise and sunset, the sky can take on hues of red, pink, and orange due to the scattering of light by atmospheric particles and the angle of the sun.
3. **Gray/White**: On overcast days or in areas with high levels of air pollution, the sky can appear gray or white due to the scattering of light by clouds or aerosols.
4. **Purple**: In some cases, the sky can appear purple during severe thunderstorms or when there are high levels of dust and water vapor in the air.
5. **Black**: At night, the sky appears black due to the absence of sunlight.

So, to answer your questio

In [13]:
import pymilvus
embeddings = openai_client.embeddings.create(
      input=[query],
      model="nvidia/llama-3.2-nv-embedqa-1b-v2",
      encoding_format="float",
      extra_body={"input_type": "query", "truncate": "END"}  
)
embeddings = [entry.embedding for entry in embeddings.data]
embeddings[0][:10], len(embeddings[0])

([-0.0237579345703125,
  0.03216552734375,
  0.02783203125,
  -0.02923583984375,
  0.012664794921875,
  0.01335906982421875,
  -0.0170135498046875,
  0.0103607177734375,
  0.0163116455078125,
  -0.033538818359375],
 2048)

In [14]:
client = pymilvus.MilvusClient(uri="http://milvus:19530")
result_ids = client.search(
    "beir_nq",
    embeddings,
    search_params={"ef": 40},
    anns_field="embedding",
    limit=10 # top_k
)

In [15]:
results_ids = [res["id"] for res in result_ids[0]]
results_ids

[579107,
 902134,
 763439,
 903868,
 903857,
 903871,
 560304,
 513668,
 741668,
 113786]

In [16]:
import pickle 
records = pickle.load(open("data/records.pickle", "rb"))
query_texts = [records[res_id]["text"] for res_id in results_ids]
query_texts

['We call both the sky and blue jeans by the same color, blue. However, clearly a pair of jeans and the sky are not the same color; moreover, the wavelengths of light reflected by the sky at every location and all the millions of blue jeans in every state of fading constantly change, and yet we somehow have a consensus of the basic form Blueness as it applies to them. Says Plato:[35][36]',
 'While the color of the sky is usually determined by Rayleigh scattering, an exception occurs at sunset and twilight. "Preferential absorption of sunlight by ozone over long horizon paths gives the zenith sky its blueness when the sun is near the horizon".[28]',
 'The bluish color is caused by an optical effect called Rayleigh scattering. The sunlit sky is blue because air scatters short-wavelength light more than longer wavelengths. Since blue light is at the short wavelength end of the visible spectrum, it is more strongly scattered in the atmosphere than long wavelength red light. The result is t

In [17]:
def get_answer(
    query,
    chunk_answers,
    endpoint:str = "https://integrate.api.nvidia.com/v1", 
    model_name: str = "meta/llama-3.3-70b-instruct",
    api_key: str = None, 
):
    api_key = api_key or os.environ.get("NVIDIA_API_KEY", None)
    api_key = "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"
    client = OpenAI(
      base_url = endpoint,
      api_key = api_key
    )
    
    completion = client.chat.completions.create(
      model=model_name,
      messages=[{"role":"user","content":f"Answer the following question. {query} With the following information: {chunk_answers}"}],
      temperature=0.2,
      top_p=0.7,
      max_tokens=1024,
      stream=True
    )
    
    for chunk in completion:
      if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")


In [18]:
get_answer(
    query,
    query_texts
)

The color of the sky is blue. This is due to an optical effect called Rayleigh scattering, where the sunlit sky appears blue because air scatters short-wavelength light (such as blue and violet) more than longer wavelengths (such as red and yellow). However, it's worth noting that the exact shade of blue can vary depending on the location, time of day, and atmospheric conditions. Additionally, during sunrise and sunset, the sky can take on hues of red, orange, and pink due to the scattering of sunlight by the atmosphere.